# Date Related Types

Pydantic offers us a number of date related types.

Let's look at a few.

In [ ]:
from pydantic import (
    BaseModel,
    ConfigDict,
    PastDatetime,
    PastDate,
    AwareDatetime,
    NaiveDatetime,
    ValidationError,
)

`PastDatetime` will restrict a field to be a valid datetime object, and also to be in the past compared to your local current time.

The docs are not exactly very forthcoming about this, but how are naive datetimes interpreted when determining if they are past or not?

Turns out they assume naive datetimes to be local, not UTC.

Let's test this out...

In [5]:
from datetime import datetime, timedelta

# import pytz

local_one_hour_ago = datetime.now() - timedelta(hours=1)
local_one_hour_ago

datetime.datetime(2026, 8, 21, 18, 47, 22, 118857)

Now, according to my timezone in AZ (we have no DST), this time, in UTC, would be:

In [ ]:
# utc_one_hour_ago = local_one_hour_ago.astimezone(pytz.utc)

But I want this datetime to be naive, not aware, so we can see how Pydantic handles it in relation to local time.

In [ ]:
# utc_naive_one_hour_ago = utc_one_hour_ago.replace(tzinfo=None)
# utc_naive_one_hour_ago

Ok, so now let's try using `PastDatetime`.

In [6]:
class Model(BaseModel):
    dt: PastDatetime

In [9]:
d = datetime.now()
d

datetime.datetime(2026, 8, 21, 19, 51, 49, 359206)

In [ ]:
m = Model(dt=local_one_hour_ago)
m

In [ ]:
m = Model(dt=utc_one_hour_ago)
m

So, both of these work fine. 

And to see that Pydantic will interpret a naive datetime as a local time:

In [ ]:
try:
    Model(dt=utc_naive_one_hour_ago)
except ValidationError as ex:
    print(ex)

Let's look at some of the other datetime types.

We have the NaiveDatetime type - this basically requires the datetime passed in to be naive.

In [ ]:
class Model(BaseModel):
    dt: NaiveDatetime

In [ ]:
local_one_hour_ago

In [ ]:
Model(dt=local_one_hour_ago)

But, if we try to pass in an aware datetime:

In [ ]:
utc_one_hour_ago

In [ ]:
try:
    Model(dt=utc_one_hour_ago)
except ValidationError as ex:
    print(ex)

And then we have the flip side of this, where we might require our datetime to be aware:

In [ ]:
class Model(BaseModel):
    dt: AwareDatetime

In [ ]:
Model(dt=utc_one_hour_ago)

But validation will fail if we pass in a naive datetime:

In [ ]:
try:
    Model(dt=local_one_hour_ago)
except ValidationError as ex:
    print(ex)

Very often, since datetime arithmetic is **not** especially easy, we end up writing custom validators and serializers.

For example, the recommended way to work with datetimes in our Python apps, is to convert everything to UTC (often as naive datetimes). This way arithmetic is much easier, since UTC does not use DST. And then we only convert to some localized (usually aware) datetime when "displaying" the data to our users.

For that, we may end up writing a custom validator that will deserialize any datetime into a UTC datetime, with some assumptions made as to what timezone should be assumed if the input datetime is naive, and do timezone conversion if it is aware. We'll circle back to this later.

In [16]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

tehran_dt = datetime(
    2026,
    8,
    21,
    14,
    0,
    tzinfo=ZoneInfo("Asia/Tehran"),
)

utc_dt = tehran_dt.astimezone(timezone.utc)

print(utc_dt)
tehran_dt

2026-08-21 10:30:00+00:00


datetime.datetime(2026, 8, 21, 14, 0, tzinfo=zoneinfo.ZoneInfo(key='Asia/Tehran'))

## mini Project Analysisi

In [12]:
from datetime import date, datetime, timezone

data = {
    "tx_date": date(2023, 1, 1),  # باید در گذشته باشد
    "created_at": datetime(2023, 1, 1, 10, 0),  # باید زمان در گذشته باشد
    "raw_time": datetime(2024, 1, 1, 12, 0),  # نباید تایم‌زون داشته باشد (Naive)
    "signed_at": datetime(
        2024, 1, 1, 12, 0, tzinfo=timezone.utc
    ),  # باید تایم‌زون داشته باشد (Aware)
}


def validate_traditional(payload):
    # ۱. چک کردن تاریخ در گذشته (PastDate)
    if payload["tx_date"] >= date.today():
        raise ValueError("tx_date must be in the past!")

    # ۲. چک کردن زمان در گذشته (PastDatetime)
    now = datetime.now(timezone.utc) if payload["created_at"].tzinfo else datetime.now()
    if payload["created_at"] >= now:
        raise ValueError("created_at must be in the past!")

    # ۳. شرط تو در تو برای چک کردن عدم وجود تایم‌زون (NaiveDatetime)
    if payload["raw_time"].tzinfo is not None:
        if payload["raw_time"].tzinfo.utcoffset(payload["raw_time"]) is not None:
            raise ValueError("raw_time must NOT have a timezone!")

    # ۴. شرط تو در تو برای اجبار به وجود تایم‌زون (AwareDatetime)
    if payload["signed_at"].tzinfo is None:
        raise ValueError("signed_at MUST have a timezone!")


validate_traditional(data)
print("Traditional: OK!")

Traditional: OK!


## Rebild By Respectfull Pydantic Datetime Types

In [13]:
from pydantic import (
    BaseModel,
    PastDate,
    PastDatetime,
    NaiveDatetime,
    AwareDatetime,
)


class TransactionSchema(BaseModel):
    tx_date: PastDate  # جایگزین شرط ۱
    created_at: PastDatetime  # جایگزین شرط ۲
    raw_time: NaiveDatetime  # جایگزین شرط ۳
    signed_at: AwareDatetime  # جایگزین شرط ۴


# تست موفقیت‌آمیز
item = TransactionSchema(
    tx_date="2023-01-01",
    created_at="2023-01-01T10:00:00Z",
    raw_time="2024-01-01T12:00:00",
    signed_at="2024-01-01T12:00:00+03:30",
)
print("Pydantic: Validated successfully!")

Pydantic: Validated successfully!


<div style="direction: rtl; text-align: center; padding: 25px 20px; border-radius: 16px; background: rgba(5, 67, 252, 0.9); border: 2px solid #00f2ff; box-shadow: 0 0 20px rgba(0, 242, 255, 0.35), inset 0 0 15px rgba(0, 242, 255, 0.2); font-family: 'Vazirmatn', Tahoma, 'Segoe UI', sans-serif;">
    <h2 style="color: #00f2ff; text-shadow: 0 0 10px #00f2ff, 0 0 25px rgba(0, 242, 255, 0.6); margin: 0 0 14px 0; font-size: 1.45em; line-height: 1.6;">
        ۳. آزمایش عملی: پایدانتیک چطور خطاها را <bdi style="font-family: Consolas, monospace;">Catch</bdi> می‌کند؟
    </h2>
    <p style="color: #f87c07; font-size: 1.05em; margin: 0; line-height: 1.8;">
        تست الف: اگر به <bdi style="color: #c1ddf8; font-family: Consolas, monospace;">NaiveDatetime</bdi> تایم‌زون (<bdi style="color: #a2c0ca; font-family: Consolas, monospace;">UTC</bdi> یا <bdi style="color: #9cc4da; font-family: Consolas, monospace;">ZoneInfo</bdi>) بدهیم چه می‌شود؟
    </p>
</div>


In [16]:
from datetime import datetime, timezone
from pydantic import ValidationError

try:
    TransactionSchema(
        tx_date="2023-01-01",
        created_at="2023-01-01T10:00:00Z",
        # خطا: مقدار دارای تایم‌زون به فیلد Naive پاس داده شده!
        raw_time=datetime.now(timezone.utc),
        signed_at="2024-01-01T12:00:00+03:30",
    )
except ValidationError as e:
    print(e.errors()[0]["msg"])

Input should not have timezone info


<div style="direction: rtl; text-align: center; padding: 25px 20px; border-radius: 16px; background: rgb(248, 5, 187); border: 2px solid #383738; box-shadow: 0 0 20px rgba(255, 0, 255, 0.35), inset 0 0 15px rgba(255, 0, 255, 0.2); font-family: 'Vazirmatn', Tahoma, 'Segoe UI', sans-serif;">
    <h2 style="color: #565a61; text-shadow: 0 0 10px #ff00ff, 0 0 25px rgba(255, 0, 255, 0.6); margin: 0; font-size: 1.45em; line-height: 1.6;">
        تست ب: اگر به <bdi style="color: #ff77ff; font-family: Consolas, monospace;">AwareDatetime</bdi> زمان خام (<bdi style="color: #ff77ff; font-family: Consolas, monospace;">Naive</bdi>) بدهیم چه می‌شود؟
    </h2>
</div>


In [19]:
try:
    TransactionSchema(
        tx_date="2023-01-01",
        created_at="2023-01-01T10:00:00Z",
        raw_time="2024-01-01T12:00:00",
        # خطا: زمان خام بدون هیچ UTC یا آفست زمانی ارسال شده!
        signed_at=datetime(2024, 1, 1, 12, 0, 0),
    )
except ValidationError as e:
    print(e.errors()[0]["msg"])

Input should have timezone info


<!-- tz > tn -->